# Imports & Setup
Import numpy, matplotlib, and all src modules (retrieval.py, injection pipeline). Set random seed and configure plot style.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

sys.path.insert(0, os.path.expanduser('~/WORK/INJECT/star-cluster-injection-pipeline'))

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.inject import inject_clusters_rubin_psf
from src.light_profiles import mag_to_flux
from src.retrieval import ClusterRetrieval

from lsst.daf.butler import Butler
from lsst.geom import Point2D

%matplotlib inline
print('Ready.')

In [ ]:
butler  = Butler('dp02', collections='2.2i/runs/DP0.2')
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd   = butler.get('deepCoadd', dataId=data_id)

CUTOUT     = 1500
BASE_IMAGE = coadd.image.array[:CUTOUT, :CUTOUT].copy()

psf_obj    = coadd.getPsf()
bbox       = coadd.getBBox()
BBOX_X_MIN = bbox.getMinX()
BBOX_Y_MIN = bbox.getMinY()

print(f'Image shape  : {BASE_IMAGE.shape}')
print(f'BBox offset  : ({BBOX_X_MIN}, {BBOX_Y_MIN})')

In [ ]:
N_GRID = 5
xs = np.linspace(BBOX_X_MIN + 100, bbox.getMaxX() - 100, N_GRID).astype(int)
ys = np.linspace(BBOX_Y_MIN + 100, bbox.getMaxY() - 100, N_GRID).astype(int)

grid_fwhm = np.full((N_GRID, N_GRID), np.nan)
for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        try:
            shape = psf_obj.computeShape(Point2D(int(x), int(y)))
            grid_fwhm[i, j] = shape.getDeterminantRadius() * 2.355
        except Exception:
            pass

valid = grid_fwhm[~np.isnan(grid_fwhm)]
PSF_FWHM_FALLBACK = float(np.median(valid)) if len(valid) > 0 else 3.5

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(grid_fwhm, origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax, label='FWHM (px)')
ax.set_title(f'PSF FWHM across coadd\nmedian={PSF_FWHM_FALLBACK:.3f} px '
             f'({PSF_FWHM_FALLBACK * 0.2:.3f} arcsec)')
plt.tight_layout(); plt.show()

In [ ]:
config = InjectionConfig(
    run_name            = 'batch_run',
    n_clusters          = 1000,   # per iteration
    seed                = 42,
    edge_buffer         = 60,
    add_noise           = True,
    use_actual_psf      = True,
    save_injected_image = False,
    output_dir          = 'outputs/batch_run',
    tract               = data_id['tract'],
    patch               = data_id['patch'],
    band                = data_id['band'],
    pixel_scale         = 0.2,
    zero_point          = 27.0,
    cluster_config = ClusterConfig(
        profile_type      = 'king',
        mag_min           = 20.0,
        mag_max           = 26.0,
        r_half_min        = 2.0,
        r_half_max        = 10.0,
        concentration_min = 5.0,
        concentration_max = 30.0,
        age_min_gyr       = 1.0,
        age_max_gyr       = 13.0,
    )
)

pipe = InjectionPipeline(config)
pipe.load_data(image=BASE_IMAGE)
print(config)

In [ ]:
def my_detector(image):
    return run_mci_detection(
        image,
        psf_fwhm_px     = PSF_FWHM_FALLBACK,
        threshold_sigma = 3.0,
        n_scales        = 5,
    )

all_injection_info, all_detections = pipe.run_batch(
    n_iterations      = 10,       # 10 runs × 1000 clusters = 10,000 total
    n_per_iter        = 1000,
    psf_obj           = psf_obj,
    bbox_x_min        = BBOX_X_MIN,
    bbox_y_min        = BBOX_Y_MIN,
    psf_fwhm_fallback = PSF_FWHM_FALLBACK,
    detector_fn       = my_detector,
    verbose           = True,
)

In [ ]:
pipe = InjectionPipeline(config)
pipe.load_data(image=BASE_IMAGE)
catalog = pipe.generate_catalog()

df_cat = pd.DataFrame([{k: v for k, v in e.items()} for e in catalog])
print(f'Catalog: {len(catalog)} clusters')
df_cat[['id','x','y','magnitude','r_half','concentration','age_gyr']].head(10)

In [ ]:
injected_image, injection_info = inject_clusters_rubin_psf(
    image             = BASE_IMAGE,
    catalog           = catalog,
    psf_obj           = psf_obj,
    bbox_x_min        = BBOX_X_MIN,
    bbox_y_min        = BBOX_Y_MIN,
    psf_fwhm_fallback = PSF_FWHM_FALLBACK,
    pixel_scale       = config.pixel_scale,
    zero_point        = config.zero_point,
    add_noise         = config.add_noise,
    use_actual_psf    = config.use_actual_psf,
    rng_seed          = config.seed,
    verbose           = True,
)

pipe.injected_image = injected_image
pipe.injection_info = injection_info

print(f'\nPSF used breakdown:')
from collections import Counter
print(Counter(i['psf_used'] for i in injection_info))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

vmin, vmax = np.percentile(BASE_IMAGE, [1, 99])

axes[0].imshow(BASE_IMAGE,     origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('Original image')

axes[1].imshow(injected_image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title(f'Injected ({len(injection_info)} clusters)')

diff = injected_image - BASE_IMAGE
axes[2].imshow(diff, origin='lower', cmap='hot',
               vmin=0, vmax=np.percentile(diff[diff > 0], 99))
axes[2].set_title('Difference (injected only)')

for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter

def run_mci_detection(image, psf_fwhm_px, threshold_sigma=3.0,
                      min_size=3, max_size=15, n_scales=5):
    """
    Multi-scale Convolution Index (MCI) detection.
    Convolves image at multiple scales and flags peaks above threshold.
    """
    from scipy.ndimage import label, maximum_filter

    background = np.median(image)
    noise      = np.std(image - gaussian_filter(image, sigma=10))

    scales  = np.linspace(psf_fwhm_px / 2.355, max_size / 2.355, n_scales)
    mci_map = np.zeros_like(image, dtype=np.float64)

    for sigma in scales:
        smoothed  = gaussian_filter(image - background, sigma=sigma)
        mci_map  += smoothed / n_scales

    # Threshold
    threshold  = threshold_sigma * noise
    peak_map   = mci_map > threshold

    # Label connected regions
    labeled, n_labels = label(peak_map)

    detections = []
    for region_id in range(1, n_labels + 1):
        mask    = labeled == region_id
        ys, xs  = np.where(mask)
        weights = mci_map[mask]
        if weights.sum() == 0:
            continue
        cx = float(np.average(xs, weights=weights))
        cy = float(np.average(ys, weights=weights))
        peak_flux = float(image[mask].sum())
        snr  = float(weights.max() / noise)
        detections.append({
            'x'    : cx,
            'y'    : cy,
            'flux' : peak_flux,
            'snr'  : snr,
        })

    return detections


detections = run_mci_detection(
    injected_image,
    psf_fwhm_px     = PSF_FWHM_FALLBACK,
    threshold_sigma = 3.0,
    n_scales        = 5,
)
print(f'MCI detections: {len(detections)}')

In [ ]:
# Cell 9 - unchanged, now uses all 10,000 injections
retrieval = ClusterRetrieval(all_injection_info, all_detections)
retrieval.match_detections(match_radius=5.0)
stats = retrieval.get_summary_statistics()
print(stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# -- magnitude completeness --
mag_bins = np.arange(config.cluster_config.mag_min,
                     config.cluster_config.mag_max + 0.5, 0.5)
bc_mag, comp_mag, err_mag = retrieval.compute_completeness('magnitude', mag_bins)

axes[0].errorbar(bc_mag, comp_mag, yerr=err_mag, fmt='o-', capsize=4)
axes[0].axhline(0.5, color='gray', ls='--', lw=1, label='50%')
axes[0].axvline(stats['mag_50_limit'], color='red', ls='--', lw=1,
                label=f"50% limit = {stats['mag_50_limit']:.2f}")
axes[0].set_xlabel('Injected magnitude')
axes[0].set_ylabel('Completeness')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Completeness vs Magnitude')
axes[0].legend()

# -- r_half completeness --
rh_bins = np.linspace(config.cluster_config.r_half_min,
                      config.cluster_config.r_half_max, 10)
bc_rh, comp_rh, err_rh = retrieval.compute_completeness('r_half', rh_bins)

axes[1].errorbar(bc_rh, comp_rh, yerr=err_rh, fmt='s-', capsize=4, color='tab:orange')
axes[1].axhline(0.5, color='gray', ls='--', lw=1, label='50%')
axes[1].axvline(stats['r_half_50_limit'], color='red', ls='--', lw=1,
                label=f"50% limit = {stats['r_half_50_limit']:.2f} px")
axes[1].set_xlabel('Half-light radius (px)')
axes[1].set_ylabel('Completeness')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Completeness vs Half-light radius')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
mag_bins2 = np.arange(config.cluster_config.mag_min,
                      config.cluster_config.mag_max + 1.0, 1.0)
rh_bins2  = np.linspace(config.cluster_config.r_half_min,
                        config.cluster_config.r_half_max, 6)

result = retrieval.compute_completeness_2d('magnitude', 'r_half', mag_bins2, rh_bins2)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(result['bin_centers1'], result['bin_centers2'],
                   result['completeness'].T,
                   cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Completeness')
ax.set_xlabel('Injected magnitude')
ax.set_ylabel('Half-light radius (px)')
ax.set_title('2D Completeness: Magnitude × Half-light radius')

# Annotate with n_injected per bin
for i in range(result['n_injected'].shape[0]):
    for j in range(result['n_injected'].shape[1]):
        n = result['n_injected'][i, j]
        if n > 0:
            ax.text(result['bin_centers1'][i], result['bin_centers2'][j],
                    str(n), ha='center', va='center', fontsize=7, color='k')

plt.tight_layout(); plt.show()

In [ ]:
matched = retrieval._matched
inj_x   = [e['x']   for e in matched]
inj_y   = [e['y']   for e in matched]
det_arr = [e.get('detected', False) for e in matched]
det_x   = [e.get('det_x', np.nan)   for e in matched]
det_y   = [e.get('det_y', np.nan)   for e in matched]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(injected_image, origin='lower', cmap='gray',
          vmin=np.percentile(injected_image, 1),
          vmax=np.percentile(injected_image, 99))

for x, y, d in zip(inj_x, inj_y, det_arr):
    color = 'lime' if d else 'red'
    ax.plot(x, y, 'o', ms=6, mec=color, mfc='none', mew=1.2)

ax.scatter([], [], marker='o', edgecolors='lime',  facecolors='none', label='Detected')
ax.scatter([], [], marker='o', edgecolors='red',   facecolors='none', label='Missed')
ax.set_title('Injection positions — green=detected, red=missed')
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
pipe.detection_catalog = detections
pipe.save_results()